In [1]:
# Cell 1: upload the dataset (skip if you load it from Google Drive)
from google.colab import files
uploaded = files.upload()   # pick movies.csv

Saving movies.csv to movies.csv


In [17]:
# Cell 2: imports and loading
import re, difflib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv("/content/drive/MyDrive/Movie Master/movies.csv", low_memory=False)
print(df.shape)
df[["title", "genres", "keywords", "director", "vote_average", "vote_count"]].head()

(4803, 24)


,title,genres,keywords,director,vote_average,vote_count
0,Avatar,Action Adventure Fantasy Science Fiction,culture clash future space war space colony so...,James Cameron,7.2,11800
1,Pirates of the Caribbean: At World's End,Adventure Fantasy Action,ocean drug abuse exotic island east india trad...,Gore Verbinski,6.9,4500
2,Spectre,Action Adventure Crime,spy based on novel secret agent sequel mi6,Sam Mendes,6.3,4466
3,The Dark Knight Rises,Action Crime Drama Thriller,dc comics crime fighter terrorist secret ident...,Christopher Nolan,7.6,9106
4,John Carter,Action Adventure Science Fiction,based on novel mars medallion space travel pri...,Andrew Stanton,6.1,2124


In [18]:
# Cell 3: cleaning
# Genres are space-separated ("Science Fiction" is two words), so match known TMDB genres.
GENRES = ["Science Fiction", "TV Movie", "Action", "Adventure", "Animation", "Comedy", "Crime",
          "Documentary", "Drama", "Family", "Fantasy", "Foreign", "History", "Horror", "Music",
          "Mystery", "Romance", "Thriller", "War", "Western"]
genre_re = re.compile("|".join(GENRES))

def parse_genres(s):
    return genre_re.findall(str(s)) if pd.notna(s) else []

for col in ["genres", "keywords", "overview", "tagline", "cast", "director"]:
    df[col] = df[col].fillna("")

df["genre_list"] = df["genres"].apply(parse_genres)
df["year"] = pd.to_datetime(df["release_date"], errors="coerce").dt.year.astype("Int64")
df = df.drop_duplicates("title").reset_index(drop=True)

# Weighted rating (IMDb formula) so a 9.0 from 3 votes doesn't beat an 8.0 from 10,000
C = df["vote_average"].mean()
m = df["vote_count"].quantile(0.60)
v, R = df["vote_count"], df["vote_average"]
df["weighted_rating"] = (v / (v + m)) * R + (m / (v + m)) * C
df["quality"] = (df["weighted_rating"] - df["weighted_rating"].min()) / (
    df["weighted_rating"].max() - df["weighted_rating"].min())
print("C =", round(C, 2), "| m =", int(m))
df[["title", "genre_list", "year", "weighted_rating"]].head()

C = 6.09 | m = 370


,title,genre_list,year,weighted_rating
0,Avatar,"[Action, Adventure, Fantasy, Science Fiction]",2009,7.166281
1,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]",2007,6.838557
2,Spectre,"[Action, Adventure, Crime]",2015,6.284077
3,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]",2012,7.541061
4,John Carter,"[Action, Adventure, Science Fiction]",2012,6.098824


In [19]:
# Cell 4: build the feature "soup" and TF-IDF matrix
def squash(name):            # "James Cameron" -> "jamescameron" (one token per person)
    return name.replace(" ", "").lower()

df["soup"] = (
    df["genre_list"].apply(lambda g: " ".join(squash(x) for x in g) + " ") * 3   # genres weigh most
    + df["keywords"].str.lower() + " "
    + df["overview"].str.lower() + " "
    + (df["director"].apply(squash) + " ") * 2
    + df["cast"].str.lower()
)

tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2, max_features=50000)
matrix = tfidf.fit_transform(df["soup"])
print("TF-IDF matrix:", matrix.shape)

TF-IDF matrix: (4800, 30680)


In [20]:
# Cell 5: Mode 1, similar movies
titles_lower = df["title"].str.lower().tolist()

def find_title(name):
    name = name.lower().strip()
    if name in titles_lower:
        return titles_lower.index(name)
    match = difflib.get_close_matches(name, titles_lower, n=1, cutoff=0.6)
    return titles_lower.index(match[0]) if match else None

def recommend_similar(name, n=5, min_votes=50):
    i = find_title(name)
    if i is None:
        print(f"'{name}' not found. Check the spelling.")
        return None
    sims = cosine_similarity(matrix[i], matrix).ravel()
    score = 0.85 * sims + 0.15 * df["quality"].values   # small nudge toward better films
    score[df["vote_count"].values < min_votes] = -1        # skip obscure titles
    score[i] = -1
    top = np.argsort(score)[::-1][:n]
    print(f"Because you liked {df.loc[i, 'title']} ({df.loc[i, 'year']}):")
    out = df.loc[top, ["title", "year", "genres", "vote_average"]].copy()
    out["score"] = score[top].round(3)
    return out.reset_index(drop=True)

recommend_similar("The Dark Knight")

Because you liked The Dark Knight (2008):


,title,year,genres,vote_average,score
0,The Dark Knight Rises,2012,Action Crime Drama Thriller,7.6,0.504
1,Batman Begins,2005,Action Crime Drama,7.5,0.472
2,The Prestige,2006,Drama Mystery Thriller,8.0,0.283
3,The Usual Suspects,1995,Drama Crime Thriller,8.1,0.282
4,Kill Bill: Vol. 2,2004,Action Crime Thriller,7.6,0.259


In [21]:
# Cell 6: mood profiles. Edit or add moods here.
MOODS = {
    "motivated": {
        "triggers": ["motivate", "inspire", "push", "hustle", "grind", "discipline", "productive",
                     "give up", "lazy", "ambition", "succeed", "success", "goal"],
        "genres": ["Drama"],
        "avoid": ["Horror", "Animation", "War"],
        "keywords": "sport sports underdog boxing boxer coach football basketball training team "
                    "competition champion comeback perseverance dream success ambition inspiration",
    },
    "happy": {
        "triggers": ["happy", "cheer", "feel good", "feel-good", "uplift", "smile", "bad day",
                     "down", "low", "wholesome", "light"],
        "genres": ["Comedy", "Family", "Animation"],
        "avoid": ["Horror", "Crime", "War", "Thriller"],
        "keywords": "friendship family heartwarming fun kindness joy feel good",
    },
    "laugh": {
        "triggers": ["laugh", "funny", "comedy", "hilarious", "silly"],
        "genres": ["Comedy"],
        "avoid": ["Horror", "War"],
        "keywords": "friends party chaos parody slapstick buddy",
    },
    "romantic": {
        "triggers": ["romant", "love", "date night", "crush", "relationship"],
        "genres": ["Romance"],
        "avoid": ["Horror", "War"],
        "keywords": "love romance relationship couple falling in love wedding",
    },
    "thrilled": {
        "triggers": ["thrill", "suspense", "edge of", "tense", "mystery", "twist", "mind"],
        "genres": ["Thriller", "Mystery", "Crime"],
        "avoid": ["Family", "Animation"],
        "keywords": "twist suspense detective investigation serial killer conspiracy",
    },
    "scared": {
        "triggers": ["scare", "scary", "horror", "spooky", "creepy", "fright"],
        "genres": ["Horror"],
        "avoid": ["Family", "Comedy", "Animation"],
        "keywords": "haunted house ghost demon possession supernatural monster",
    },
    "adrenaline": {
        "triggers": ["action", "adrenaline", "pumped", "hype", "energy", "explosive", "fight"],
        "genres": ["Action", "Adventure"],
        "avoid": ["Romance", "Documentary"],
        "keywords": "fight chase revenge mission explosion hero battle",
    },
    "thoughtful": {
        "triggers": ["think", "deep", "philosoph", "smart", "reflect", "existential", "curious"],
        "genres": ["Science Fiction", "Drama", "Mystery"],
        "avoid": ["Comedy", "Family"],
        "keywords": "reality identity memory time travel artificial intelligence dystopia existence",
    },
    "emotional": {
        "triggers": ["cry", "emotional", "tearjerker", "sad movie", "grief", "heartbreak"],
        "genres": ["Drama", "Romance"],
        "avoid": ["Horror", "Action", "Comedy"],
        "keywords": "grief loss death love family sacrifice terminal illness",
    },
    "adventurous": {
        "triggers": ["adventure", "travel", "explore", "escape", "wanderlust", "journey", "fantasy"],
        "genres": ["Adventure", "Fantasy"],
        "avoid": ["Horror"],
        "keywords": "journey quest travel exploration magic treasure self discovery",
    },
}

In [22]:
# Cell 7: Mode 2, recommend by mood
genre_sets = df["genre_list"].apply(set)

def detect_moods(text):
    text = text.lower()
    hits = {m: sum(t in text for t in p["triggers"]) for m, p in MOODS.items()}
    hits = {m: h for m, h in hits.items() if h}
    return sorted(hits, key=hits.get, reverse=True)[:2]   # up to 2 moods can blend

def recommend_by_mood(text, n=5, min_votes=300):
    moods = detect_moods(text)
    query = " ".join(MOODS[m]["keywords"] for m in moods) if moods else text
    want = set().union(*[MOODS[m]["genres"] for m in moods]) if moods else set()
    avoid = set().union(*[MOODS[m]["avoid"] for m in moods]) if moods else set()

    kw = cosine_similarity(tfidf.transform([query.lower()]), matrix).ravel()
    kw = kw / kw.max() if kw.max() > 0 else kw
    genre = genre_sets.apply(lambda g: len(g & want) / len(want) if want else 0).values
    penalty = genre_sets.apply(lambda g: 0.5 if g & avoid else 0).values

    score = 0.25 * genre + 0.50 * kw + 0.25 * df["quality"].values - penalty
    score[df["vote_count"].values < min_votes] = -1        # skip obscure titles
    top = np.argsort(score)[::-1][:n]
    print("Detected mood:", ", ".join(moods) if moods else "none (matched your words directly)")
    out = df.loc[top, ["title", "year", "genres", "vote_average"]].copy()
    out["score"] = score[top].round(3)
    return out.reset_index(drop=True)

recommend_by_mood("I need motivation")

Detected mood: none (matched your words directly)


,title,year,genres,vote_average,score
0,The Full Monty,1997,Comedy,6.8,0.585
1,"Lock, Stock and Two Smoking Barrels",1998,Comedy Crime,7.5,0.565
2,The Avengers,2012,Science Fiction Action Adventure,7.4,0.516
3,How to Be Single,2016,Comedy Romance,5.9,0.509
4,American Pie,1999,Comedy Romance,6.4,0.490


In [23]:
recommend_by_mood("had a bad day, cheer me up")

Detected mood: happy


,title,year,genres,vote_average,score
0,Inside Out,2015,Drama Comedy Animation Family,8.0,0.852
1,That Awkward Moment,2014,Comedy Romance,6.1,0.676
2,The Lego Movie,2014,Adventure Animation Comedy Family Fantasy,7.5,0.654
3,Toy Story,1995,Animation Comedy Family,7.7,0.645
4,The Boxtrolls,2014,Animation Comedy Family Fantasy,6.6,0.629


In [24]:
recommend_by_mood("something scary tonight")

Detected mood: scared


,title,year,genres,vote_average,score
0,Insidious: Chapter 2,2013,Horror Thriller,6.5,0.878
1,The Others,2001,Horror Drama Mystery Thriller,7.4,0.687
2,Sinister 2,2015,Horror,5.4,0.640
3,Stir of Echoes,1999,Horror Mystery Thriller,6.5,0.635
4,Poltergeist,1982,Horror,7.0,0.579


In [25]:
recommend_by_mood("funny romantic movie for date night")

Detected mood: romantic, laugh


,title,year,genres,vote_average,score
0,Monster-in-Law,2005,Romance Comedy,5.6,0.739
1,The Princess Diaries 2: Royal Engagement,2004,Comedy Drama Family Romance,6.0,0.699
2,Moonrise Kingdom,2012,Comedy Drama Romance,7.6,0.689
3,Forrest Gump,1994,Comedy Drama Romance,8.2,0.677
4,Annie Hall,1977,Comedy Drama Romance,7.8,0.675


In [26]:
# Cell 8: try your own
recommend_similar(input("Want to find similar movies of what you like?? Type in the movie name: "))

Want to find similar movies of what you like?? Type in the movie name: Avatar
Because you liked Avatar (2009):


,title,year,genres,vote_average,score
0,X-Men: Days of Future Past,2014,Action Adventure Fantasy Science Fiction,7.5,0.385
1,Man of Steel,2013,Action Adventure Fantasy Science Fiction,6.5,0.349
2,Superman,1978,Action Adventure Fantasy Science Fiction,6.9,0.329
3,Superman II,1980,Action Adventure Fantasy Science Fiction,6.5,0.312
4,Small Soldiers,1998,Comedy Adventure Fantasy Science Fiction Action,6.2,0.308


In [27]:
recommend_by_mood(input("How are you feeling? "))

How are you feeling? Motivated
Detected mood: motivated


,title,year,genres,vote_average,score
0,The Hurricane,1999,Drama,7.2,0.824
1,The Longest Yard,2005,Comedy Drama,6.2,0.802
2,Remember the Titans,2000,Drama,7.4,0.796
3,Any Given Sunday,1999,Drama,6.8,0.765
4,Ali,2001,Drama,6.7,0.685
